<a href="https://colab.research.google.com/github/maikol0629/dl_voice_command_cl/blob/main/02_preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocesamiento y Extracción de Características para Clasificación de Comandos de Voz

## Introducción

Este notebook documenta el pipeline de preprocesamiento de audio:
- Carga de archivos `.npy` con forma de onda
- Normalización de longitud (padding/truncation a 16000 muestras)
- Extracción de características tiempo-frecuencia (MFCC, Mel-spectrogramas)
- Data augmentation (ruido, SpecAugment)
- División train/val estratificada

Discutimos también los dos paradigmas principales para el procesamiento de voz:
1. **Paradigma de múltiples instancias**: ventanas asumidas independientes (CNN 2D sobre espectrogramas)
2. **Paradigma de secuencias**: ventanas con dependencia temporal (CRNN, Transformers)

In [ ]:
# Colab setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
        os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    except Exception:
        pass
        
    !pip install -q kagglehub torch torchaudio pandas numpy matplotlib scikit-learn librosa soundfile audioread numba
    import kagglehub
    import shutil
    
    try:
        data_path = kagglehub.competition_download("voice-commands-classification-2026")
    except Exception as e:
        print("Error al descargar de Kaggle. Asegúrate de configurar KAGGLE_USERNAME y KAGGLE_KEY en tus Colab Secrets.")
        raise e
        
    if not os.path.exists('data'):
        os.symlink(data_path, 'data')
        
    if not os.path.exists('dl_voice_command_cl'):
        !git clone https://github.com/maikol0629/dl_voice_command_cl.git
        
    for f in ['train_metadata.csv', 'test_metadata.csv']:
        src = os.path.join('dl_voice_command_cl', f)
        if os.path.exists(src) and not os.path.exists(f):
            shutil.copy(src, '.')
            
        DATA_PATH = os.path.join(data_path, 'train')
        AUDIO_DIR = os.path.join(DATA_PATH, 'train', 'train')
else:
    import os
    DATA_PATH = './data/train'
    AUDIO_DIR = os.path.join(DATA_PATH, 'train', 'train')


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

SAMPLE_RATE = 16000
DURATION = 1.0
MAX_SAMPLES = int(SAMPLE_RATE * DURATION)
N_MELS = 64
N_MFCC = 120
N_FFT = 1024
HOP_LENGTH = 256
N_CLASSES = 35
VAL_SIZE = 0.2
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Carga de Metadatos

El dataset incluye un archivo `metadata.csv` con dos columnas: `file_name` (nombre del archivo .npy) y `label` (comando de voz, 35 clases).

In [ ]:
metadata_path = os.path.join(DATA_PATH, 'metadata.csv')
df = pd.read_csv(metadata_path)
print(f"Registros: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nPrimeras filas:\n{df.head()}")
print(f"\nClases ({df['label'].nunique()}):")
print(df['label'].value_counts())

In [ ]:
# Codificar etiquetas
label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['label'])
class_names = label_encoder.classes_
print(f"Clases codificadas: {dict(zip(class_names, range(N_CLASSES)))}")

## 2. Dataset Class: Carga y Normalización de Longitud

Los archivos `.npy` contienen arrays NumPy de forma `(1, N)` donde N varía entre ~3400 y 16000 muestras.
Estrategia: padding con ceros (o truncamiento) a 16000 muestras.

In [ ]:
class VoiceCommandsDataset(Dataset):
    def __init__(self, df, audio_dir, max_samples=MAX_SAMPLES, transform=None):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.max_samples = max_samples
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _normalize_length(self, audio):
        if audio.shape[0] > self.max_samples:
            return audio[:self.max_samples]
        if audio.shape[0] < self.max_samples:
            padding = self.max_samples - audio.shape[0]
            return np.pad(audio, (0, padding), mode='constant')
        return audio

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = self.audio_dir / row['file_name']
        arr = np.load(file_path, allow_pickle=False)
        if isinstance(arr, np.lib.npyio.NpzFile):
            key = 'audio' if 'audio' in arr.files else arr.files[0]
            arr = arr[key]
        audio = np.asarray(arr).reshape(-1).astype(np.float32)
        audio = self._normalize_length(audio)
        label = int(row['label_id'])
        waveform = torch.tensor(audio, dtype=torch.float32)
        if self.transform:
            features = self.transform(waveform.unsqueeze(0))
            return features, label
        return waveform, label

## 3. Caracterización de la Señal de Voz

### Opción A: Mel-Spectrograma
Transformada de Fourier de tiempo corto (STFT) → escala Mel → decibeles → normalización por instancia.
**Entrada esperada para CNN 2D.**

### Opción B: MFCC (Mel-Frequency Cepstral Coefficients)
Los coeficientes MFCC capturan la forma espectral de la señal. Son el estándar en reconocimiento de voz.
**Entrada esperada para Transformer.**

### Opción C: Forma de onda cruda
La señal de audio sin procesar. Procesada directamente por Conv1D + LSTM.
**Entrada esperada para CRNN.**

In [ ]:
# Transformación A: Mel-Spectrograma
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    ),
    T.AmplitudeToDB()
)

# Transformación B: MFCC
mfcc_transform = T.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    log_mels=True,
    melkwargs={
        'n_fft': N_FFT,
        'hop_length': HOP_LENGTH,
        'n_mels': N_MFCC
    }
)

print("Transformaciones definidas")
print(f"Mel: {N_MELS} bands, hop={HOP_LENGTH}")
print(f"MFCC: {N_MFCC} coefficients, hop={HOP_LENGTH}")

In [ ]:
# Visualización: waveform + mel-spectrograma de una muestra
sample = df.iloc[0]
audio_path = Path(AUDIO_DIR) / sample['file_name']
arr = np.load(audio_path).reshape(-1).astype(np.float32)
arr = arr[:MAX_SAMPLES] if len(arr) > MAX_SAMPLES else np.pad(arr, (0, max(0, MAX_SAMPLES - len(arr))))

waveform_t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
mel_t = mel_transform(waveform_t)
mfcc_t = mfcc_transform(waveform_t)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
axes[0].plot(arr[:3200])
axes[0].set_title(f'Waveform - {sample["label"]} (primeros 3200 samples)')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Amplitud')

axes[1].imshow(mel_t.squeeze().numpy(), aspect='auto', origin='lower')
axes[1].set_title('Mel-Spectrograma (dB)')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Mel Band')

axes[2].imshow(mfcc_t.squeeze().numpy(), aspect='auto', origin='lower')
axes[2].set_title('MFCC')
axes[2].set_xlabel('Frame')
axes[2].set_ylabel('Coefficient')

plt.tight_layout()
plt.show()

print(f"Waveform shape: {arr.shape}")
print(f"Mel shape: {mel_t.shape}")
print(f"MFCC shape: {mfcc_t.shape}")

## 4. Data Augmentation

Para mejorar la robustez ante ataques adversariales, aplicamos aumento de datos:
- **Ruido Gaussiano**: simula perturbaciones pequeñas
- **SpecAugment**: enmascaramiento de bandas de frecuencia y tramos de tiempo (útil para CNN/Transformer)
- **Time Stretching**: variación leve de velocidad

In [ ]:
class AddGaussianNoise(nn.Module):
    def __init__(self, noise_level=0.005):
        super().__init__()
        self.noise_level = noise_level

    def forward(self, x):
        if self.training and self.noise_level > 0:
            noise = torch.randn_like(x) * self.noise_level
            return x + noise
        return x

class SpecAugment(nn.Module):
    def __init__(self, freq_mask=15, time_mask=15):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask

    def forward(self, x):
        if self.training:
            x = T.FrequencyMasking(self.freq_mask)(x)
            x = T.TimeMasking(self.time_mask)(x)
        return x

noise_aug = AddGaussianNoise(0.005)
spec_augment = SpecAugment(freq_mask=8, time_mask=8)
print("Data augmentation listo")

## 5. Split Train/Val y DataLoaders

División estratificada 80/20 para mantener la proporción de clases.

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df['label_id']
)

train_dataset = VoiceCommandsDataset(train_df, AUDIO_DIR)
val_dataset = VoiceCommandsDataset(val_df, AUDIO_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"Train: {len(train_dataset)} muestras")
print(f"Val: {len(val_dataset)} muestras")
print(f"Clases: {N_CLASSES}")

## 6. Discusión: Paradigma de Procesamiento

### Paradigma de Múltiples Instancias (Ventanas Independientes)
- Se divide el espectrograma en ventanas de tiempo fijas
- Cada ventana se trata como instancia independiente
- Adecuado para **CNN 2D** que operan sobre el espectrograma como una imagen
- **Ventaja**: simple, computacionalmente eficiente
- **Desventaja**: ignora dependencias temporales entre ventanas

### Paradigma de Secuencias (Dependencia Temporal)
- Las ventanas se tratan como pasos temporales de una secuencia
- Modelos recurrentes (LSTM, GRU) o Transformers capturan la evolución temporal
- **Ventaja**: modela la dinámica temporal del habla
- **Desventaja**: mayor complejidad, más parámetros

### Decisión Arquitectural
En este proyecto exploramos **los tres paradigmas**:

| Modelo | Caracterización | Paradigma |
|--------|----------------|-----------|
| CNN Baseline | Mel-spectrograma | Múltiples instancias (ventanas independientes) |
| CRNN (Conv1D + BiLSTM) | Forma de onda cruda | Secuencias (dependencia temporal) |
| Spectrogram Transformer | MFCC + RoPE | Secuencias (auto-atención) |